# Read and validate data

This notebook is designed just for convenience

In [21]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find a repository containing data/")

def compute_scalar(value):
    value = value.compute() if hasattr(value, "compute") else value
    return value.item() if hasattr(value, "item") else value

def inspect_store(kind: str, path: Path) -> list[dict]:
    relative_path = path.relative_to(REPO_ROOT)
    print(f"\n{kind.upper()}: {relative_path}")
    opener = xr.open_zarr(path, consolidated=False) if kind == "zarr" else xr.open_dataset(path, decode_cf=True, mask_and_scale=True)
    with opener as ds:
        print("Dimensions:", dict(ds.sizes))
        print("Coordinates:", list(ds.coords))
        print("Data variables:", list(ds.data_vars))
        print("Global attributes:", dict(ds.attrs))
        print("Dataset encoding keys:", sorted(ds.encoding))
        rows = []
        for name, da in ds.variables.items():
            size = int(da.size)
            missing = int(compute_scalar(da.isnull().sum())) if size else 0
            row = {"store_type": kind, "path": str(relative_path), "name": name,
                   "role": "coordinate" if name in ds.coords else "data",
                   "dtype": str(da.dtype), "dims": tuple(da.dims), "shape": tuple(da.shape),
                   "size": size, "missing": missing,
                   "missing_pct": round(100 * missing / size, 3) if size else 0.0,
                   "attrs": dict(da.attrs)}
            if size and da.dtype.kind in "iufc":
                row["finite"] = int(compute_scalar(np.isfinite(da).sum()))
                if row["finite"]:
                    row["min"] = compute_scalar(da.min(skipna=True))
                    row["max"] = compute_scalar(da.max(skipna=True))
            rows.append(row)
            print(f"  {name}: missing={missing}/{size} ({row['missing_pct']}%), dtype={row['dtype']}, shape={row['shape']}")
        return rows

REPO_ROOT = find_repo_root()
DATA_ROOT = REPO_ROOT / "data"
raw_files = sorted((DATA_ROOT / "raw").rglob("*.nc"))
interim_files = sorted((DATA_ROOT / "interim").rglob("*.nc"))
zarr_stores = sorted(path for path in (DATA_ROOT / "zarr").iterdir()
                     if path.is_dir() and ((path / "zarr.json").exists() or (path / ".zgroup").exists()))
print(f"Found {len(raw_files)} raw, {len(interim_files)} interim, and {len(zarr_stores)} Zarr stores.")

Found 30 raw, 30 interim, and 1 Zarr stores.


## RAW data

In [17]:
print("=" * 100)
print(f"RAW DATA ({len(raw_files)} files)")
print("=" * 100)
raw_rows = []
for raw_path in raw_files:
    raw_rows.extend(inspect_store("raw", raw_path))
raw_variables = pd.DataFrame(raw_rows)
display(raw_variables.sort_values(["path", "role", "name"]))

RAW DATA (30 files)

RAW: data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_METREF_MSG-Disk_202006010000.nc
Dimensions: {'time': 1, 'lat': 3201, 'lon': 3201}
Coordinates: ['lat', 'lon', 'time']
Data variables: ['METREF', 'crs', 'quality_flag']
Global attributes: {'date_created': '2020-10-23T07:20:57Z', 'algorithm_version': '1.3.2', 'base_algorithm_version': '1.0.3', 'processing_level': '03', 'platform': 'MSG4', 'sensor': 'SEVI', 'parent_product_name': 'METREF;QFLAGS;-;-', 'time_coverage_start': '2020-06-01T00:00:00Z', 'image_reference_time': '2020-06-01T00:00:00Z', 'name': 'METREF', 'region_name': 'MSG-Disk', 'institution': 'IM-PT', 'archive_facility': 'IPMA', 'license': 'Data is freely available for users registered at the LSA SAF website - lsa-saf.eumetsat.int', 'history': '2020-10-23T07:20:57Z - NetCDF reprojected product ; 2020-06-02T00:00:24Z Original product file created: HDF5_LSASAF_MSG_METREF_MSG-Disk_202006010000', 'geospatial_lat_resolution': ' 0.05', 'geospatial_lon_resolution': ' 

,store_type,path,name,role,dtype,dims,shape,size,missing,missing_pct,attrs,finite,min,max
3,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,lat,coordinate,float32,"(lat,)","(3201,)",3201,0,0.000,"{'units': 'degrees_north', 'standard_name': 'l...",3201.0,-80.0,80.00
4,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,lon,coordinate,float32,"(lon,)","(3201,)",3201,0,0.000,"{'units': 'degrees_east', 'standard_name': 'lo...",3201.0,-80.0,80.00
5,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,time,coordinate,datetime64[ns],"(time,)","(1,)",1,0,0.000,"{'long_name': 'time', 'axis': 'T'}",NaN,NaN,NaN
0,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,METREF,data,float64,"(time, lat, lon)","(1, 3201, 3201)",10246401,7568883,73.869,"{'long_name': 'METREF', 'units': 'mm/day', 'gr...",2677518.0,0.0,6.59
1,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,crs,data,|S1,(),(),1,0,0.000,"{'grid_mapping_name': 'latitude_longitude', 'l...",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,lon,coordinate,float32,"(lon,)","(3201,)",3201,0,0.000,"{'units': 'degrees_east', 'standard_name': 'lo...",3201.0,-80.0,80.00
179,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,time,coordinate,datetime64[ns],"(time,)","(1,)",1,0,0.000,"{'long_name': 'time', 'axis': 'T'}",NaN,NaN,NaN
174,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,METREF,data,float64,"(time, lat, lon)","(1, 3201, 3201)",10246401,7562111,73.803,"{'long_name': 'METREF', 'units': 'mm/day', 'gr...",2684290.0,0.0,6.69
175,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,crs,data,|S1,(),(),1,0,0.000,"{'grid_mapping_name': 'latitude_longitude', 'l...",NaN,NaN,NaN


## INTERIM data

In [22]:
print("=" * 100)
print(f"INTERIM DATA ({len(interim_files)} files)")
print("=" * 100)
interim_rows = []
for interim_path in interim_files:
    interim_rows.extend(inspect_store("interim", interim_path))
interim_variables = pd.DataFrame(interim_rows)
display(interim_variables.sort_values(["path", "role", "name"]))

INTERIM DATA (30 files)

INTERIM: data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF_MSG-Disk_202006010000_processed.nc
Dimensions: {'time': 1, 'lat': 91, 'lon': 121}
Coordinates: ['time', 'lon', 'lat']
Data variables: ['METREF', 'quality_flag']
Global attributes: {'CDI': 'Climate Data Interface version 2.4.0 (https://mpimet.mpg.de/cdi)', 'Conventions': 'CF-1.6', 'institution': 'IM-PT', 'date_created': '2020-10-23T07:20:57Z', 'algorithm_version': '1.3.2', 'base_algorithm_version': '1.0.3', 'processing_level': '03', 'platform': 'MSG4', 'sensor': 'SEVI', 'parent_product_name': 'METREF;QFLAGS;-;-', 'time_coverage_start': '2020-06-01T00:00:00Z', 'image_reference_time': '2020-06-01T00:00:00Z', 'name': 'METREF', 'region_name': 'MSG-Disk', 'archive_facility': 'IPMA', 'license': 'Data is freely available for users registered at the LSA SAF website - lsa-saf.eumetsat.int', 'history': 'Thu Oct 15 13:51:02 2026: cdo -L -f nc4 -O merge /mnt/c/Users/b1113715/Downloads/tutorial-repo/data/interim/2020/06

,store_type,path,name,role,dtype,dims,shape,size,missing,missing_pct,attrs,finite,min,max
4,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,lat,coordinate,float64,"(lat,)","(91,)",91,0,0.000,"{'standard_name': 'latitude', 'long_name': 'la...",91.0,12.250000,16.750000
3,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,lon,coordinate,float64,"(lon,)","(121,)",121,0,0.000,"{'standard_name': 'longitude', 'long_name': 'l...",121.0,-17.250000,-11.250000
2,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,time,coordinate,datetime64[ns],"(time,)","(1,)",1,0,0.000,"{'standard_name': 'time', 'long_name': 'time',...",NaN,NaN,NaN
0,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,METREF,data,float32,"(time, lat, lon)","(1, 91, 121)",11011,1039,9.436,"{'long_name': 'METREF', 'units': 'mm/day'}",9972.0,1.990010,5.629999
1,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,quality_flag,data,float64,"(time, lat, lon)","(1, 91, 121)",11011,0,0.000,{'long_name': 'QFLAGS'},11011.0,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,lat,coordinate,float64,"(lat,)","(91,)",91,0,0.000,"{'standard_name': 'latitude', 'long_name': 'la...",91.0,12.250000,16.750000
148,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,lon,coordinate,float64,"(lon,)","(121,)",121,0,0.000,"{'standard_name': 'longitude', 'long_name': 'l...",121.0,-17.250000,-11.250000
147,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,time,coordinate,datetime64[ns],"(time,)","(1,)",1,0,0.000,"{'standard_name': 'time', 'long_name': 'time',...",NaN,NaN,NaN
145,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,METREF,data,float32,"(time, lat, lon)","(1, 91, 121)",11011,1039,9.436,"{'long_name': 'METREF', 'units': 'mm/day'}",9972.0,1.640006,5.619974


## ZARR data

In [19]:
print("=" * 100)
print(f"ZARR DATA ({len(zarr_stores)} stores)")
print("=" * 100)
zarr_rows = []
for zarr_path in zarr_stores:
    zarr_rows.extend(inspect_store("zarr", zarr_path))
zarr_variables = pd.DataFrame(zarr_rows)
display(zarr_variables.sort_values(["path", "role", "name"]))

ZARR DATA (1 stores)

ZARR: data\zarr\metref_sample_1month.zarr
Dimensions: {'time': 30, 'lat': 552, 'lon': 576}
Coordinates: ['lat', 'lon', 'time']
Data variables: ['quality_flag', 'METREF']
Global attributes: {'CDI': 'Climate Data Interface version 2.4.0 (https://mpimet.mpg.de/cdi)', 'Conventions': 'CF-1.6', 'institution': 'IM-PT', 'date_created': '2020-10-23T07:20:57Z', 'algorithm_version': '1.3.2', 'base_algorithm_version': '1.0.3', 'processing_level': '03', 'platform': 'MSG4', 'sensor': 'SEVI', 'parent_product_name': 'METREF;QFLAGS;-;-', 'time_coverage_start': '2020-06-01T00:00:00Z', 'image_reference_time': '2020-06-01T00:00:00Z', 'name': 'METREF', 'region_name': 'MSG-Disk', 'archive_facility': 'IPMA', 'license': 'Data is freely available for users registered at the LSA SAF website - lsa-saf.eumetsat.int', 'history': 'Wed Oct 14 10:45:01 2026: cdo -L -f nc4 -O -remapbil,/mnt/c/Users/b1113715/Downloads/tutorial-repo/code/preprocess_utils/grid.txt /tmp/tmp.nhKZV1CDqC /mnt/c/Users/b1

,store_type,path,name,role,dtype,dims,shape,size,missing,missing_pct,attrs,finite,min,max
2,zarr,data\zarr\metref_sample_1month.zarr,lat,coordinate,float64,"(lat,)","(552,)",552,0,0.000,"{'standard_name': 'latitude', 'long_name': 'la...",552.0,-10.944114,16.605886
3,zarr,data\zarr\metref_sample_1month.zarr,lon,coordinate,float64,"(lon,)","(576,)",576,0,0.000,"{'standard_name': 'longitude', 'long_name': 'l...",576.0,-17.180733,11.569267
4,zarr,data\zarr\metref_sample_1month.zarr,time,coordinate,datetime64[ns],"(time,)","(30,)",30,0,0.000,"{'standard_name': 'time', 'long_name': 'time',...",NaN,NaN,NaN
1,zarr,data\zarr\metref_sample_1month.zarr,METREF,data,float32,"(time, lat, lon)","(30, 552, 576)",9538560,5756325,60.348,"{'long_name': 'METREF', 'units': 'mm/day'}",3782235.0,0.710000,6.086871
0,zarr,data\zarr\metref_sample_1month.zarr,quality_flag,data,float64,"(time, lat, lon)","(30, 552, 576)",9538560,369,0.004,{'long_name': 'QFLAGS'},9538191.0,-4.000000,2.000000


## Missing-value comparison

In [20]:
all_variables = pd.concat([raw_variables, interim_variables, zarr_variables], ignore_index=True)
display(all_variables.loc[all_variables["missing"] > 0,
                          ["store_type", "path", "name", "role", "missing", "size", "missing_pct"]]
        .sort_values(["store_type", "missing_pct"], ascending=[True, False]))

,store_type,path,name,role,missing,size,missing_pct
265,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,METREF,data,194821,317952,61.274
180,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,METREF,data,191776,317952,60.316
185,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,METREF,data,191776,317952,60.316
190,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,METREF,data,191776,317952,60.316
195,interim,data\interim\2020\06\NETCDF4_LSASAF_MSG_METREF...,METREF,data,191776,317952,60.316
...,...,...,...,...,...,...,...
164,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,quality_flag,data,2204174,10246401,21.512
170,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,quality_flag,data,2204174,10246401,21.512
176,raw,data\raw\METREF\2020\06\NETCDF4_LSASAF_MSG_MET...,quality_flag,data,2204174,10246401,21.512
331,zarr,data\zarr\metref_sample_1month.zarr,METREF,data,5756325,9538560,60.348
